# Chapter 05. 코사인 유사도로 비슷한 책 추천하기

Chapter 04에서는 도서 제목을 TF-IDF 벡터로 바꾸고 Naive Bayes 모델로 **분야를 분류**했습니다.

이번 Chapter에서는 같은 TF-IDF 표현을 **추천**에 활용합니다.

전체 흐름은 다음과 같습니다.

**데이터 준비 → 상품명 TF-IDF → 기준 도서 선택 → 전체 도서와 Cosine Similarity → 유사도 내림차순 정렬 → 자기 자신 제외 → Top 5 → 추천 함수 → 결과 검증 → CSV 저장 → Streamlit 연결 준비**

이번 Chapter의 목표는 복잡한 추천 알고리즘을 만드는 것이 아닙니다.

> **도서 제목을 벡터로 표현하고, 선택한 도서와 다른 도서의 유사도를 계산하여 Top 5를 추천하는 전체 흐름을 이해하는 것**이 핵심입니다.

이번 Notebook도 앞 Chapter들과 같은 방식으로 작성합니다.

**해야 할 일 이해 → AI에게 질문 → AI 답변 확인 → 주석 코드 실행 → 출력값 직접 확인 → 검증 → 자세한 Markdown 정리**

### 이번 Chapter에서 꼭 구분할 것

이번 추천은 사용자의 구매·클릭·평점 정보를 이용하는 개인화 추천이 아닙니다.

**도서 제목 텍스트가 서로 얼마나 비슷한지**를 기준으로 하는 콘텐츠 기반 추천입니다.

## 학습 목표

이번 실습이 끝나면 다음을 설명할 수 있어야 합니다.

- 콘텐츠 기반 추천이 무엇인지 설명할 수 있다.
- 도서 제목을 TF-IDF 벡터로 변환할 수 있다.
- 코사인 유사도의 의미를 설명할 수 있다.
- 선택한 도서와 전체 도서의 유사도를 계산할 수 있다.
- 자기 자신을 추천 결과에서 제외할 수 있다.
- 유사도 순으로 정렬하여 Top 5를 만들 수 있다.
- 추천 로직을 함수로 만들 수 있다.
- 추천 결과를 원본 데이터와 비교해 검증할 수 있다.
- 제목 기반 추천의 한계를 설명할 수 있다.
- Chapter 06의 Streamlit 앱과 연결할 수 있다.

## 실습 1. 라이브러리 준비하기

### AI에게 질문

> Python과 머신러닝을 처음 배우고 있습니다.  
> TF-IDF와 코사인 유사도를 이용해 도서 제목 기반 추천을 만들려고 합니다.
>
> 다음 라이브러리를 불러오는 코드를 작성해 주세요.
>
> - numpy
> - pandas
> - TfidfVectorizer
> - cosine_similarity
>
> 현재 Notebook이 사용하는 Python 버전과 scikit-learn 버전도 확인하고 싶습니다.

### AI 답변

추천에 필요한 표 데이터는 pandas로 다루고, 벡터 계산 확인에는 numpy를 사용합니다. TF-IDF는 `TfidfVectorizer`, 벡터 간 유사도는 `cosine_similarity`를 사용합니다.

In [ ]:
# 현재 Notebook이 실제로 사용하는 Python 환경을 확인합니다.
import sys

print("Python 실행 파일:")
print(sys.executable)

print("\nPython 버전:")
print(sys.version)

# 수치 계산
import numpy as np

# 표 형태 데이터 처리
import pandas as pd

# 텍스트를 TF-IDF 벡터로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# 두 벡터의 코사인 유사도를 계산
from sklearn.metrics.pairwise import cosine_similarity

# 표와 Markdown을 보기 좋게 출력
from IPython.display import display, Markdown

# scikit-learn 버전 확인
import sklearn

print("\nscikit-learn 버전:", sklearn.__version__)

### 실습 1 결과 확인 및 정리

이 셀이 오류 없이 실행되면 추천 실습에 필요한 기본 라이브러리를 사용할 수 있습니다.

만약 `ModuleNotFoundError: No module named 'sklearn'`이 발생하면 현재 Notebook이 사용하는 Python과 패키지를 설치한 Python이 다른지 먼저 확인합니다.

현재 Notebook 커널에 설치하려면 코드 셀에서 다음처럼 실행할 수 있습니다.

```python
import sys
!{sys.executable} -m pip install scikit-learn
```

설치가 끝나면 Kernel을 재시작하고 다시 import합니다.

## 실습 2. 데이터 불러오기

### AI에게 질문

> Chapter 01에서 만든 `book_bestseller_clean.csv`를 불러오고 싶습니다.  
> 현재 프로젝트 루트는 `C:\dev\llm-data-analysis-course`이고 파일은 `notebooks/book-text-ml` 폴더에 있습니다.
>
> 다음을 확인하는 코드를 작성해 주세요.
>
> 1. 파일 존재 여부  
> 2. 데이터 크기  
> 3. 컬럼 이름  
> 4. 상품명 결측치 개수  
> 5. 상품명과 함께 저자/인물, 출판사, 분야가 있다면 앞의 10행 표시

### AI 답변

추천에서 가장 중요한 입력은 `상품명`입니다. Chapter 01에서 컬럼 이름을 `저자`로 바꿨을 수도 있으므로, 부가 정보는 실제 존재하는 컬럼만 선택하도록 작성합니다.

In [ ]:
# 파일 경로 확인을 위해 Path를 불러옵니다.
from pathlib import Path

# 프로젝트 루트를 기준으로 한 CSV 경로입니다.
DATA_PATH = Path("notebooks/book-text-ml/book_bestseller_clean.csv")

print("CSV 파일 존재:", DATA_PATH.exists())
print("CSV 경로:", DATA_PATH)

# CSV 파일을 불러옵니다.
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

print("\n데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

# Chapter 01에서 '인물'을 '저자'로 바꿨을 수 있으므로
# 실제 존재하는 컬럼만 골라 확인합니다.
preview_columns = [
    col
    for col in ["상품명", "저자", "인물", "출판사", "분야"]
    if col in df_books.columns
]

df_books[preview_columns].head(10)

### 실습 2 결과 확인 및 정리

추천에서 반드시 필요한 컬럼은 **상품명**입니다.

추천 결과를 사람이 검토하기 쉽게 하기 위해 가능하면 다음 정보도 같이 봅니다.

- 상품명
- 저자 또는 인물
- 출판사
- 분야

이번 Notebook은 Chapter 01에서 `인물`을 `저자`로 바꾼 경우에도 작동하도록 **실제 존재하는 컬럼만 자동으로 선택**합니다.

파일이 열렸다는 사실만 보지 말고 한글과 제목이 정상적으로 보이는지도 확인합니다.

## 실습 3. 추천용 데이터 준비하기

### AI에게 질문

> 추천용 DataFrame을 만들고 싶습니다.
>
> 조건:
> 1. 원본을 copy  
> 2. 상품명 결측치는 빈 문자열  
> 3. 문자열 변환  
> 4. 앞뒤 공백 제거  
> 5. 빈 상품명 제외  
> 6. index를 0부터 다시 설정  
> 7. 전처리 전후 도서 수 비교
>
> 특히 DataFrame index와 TF-IDF matrix 행 번호가 같은 도서를 가리켜야 하는 이유도 설명해 주세요.

### AI 답변

추천에서는 index를 이용해 TF-IDF 행을 선택하므로 DataFrame의 index와 matrix row 번호를 맞추는 것이 매우 중요합니다.

In [ ]:
# 원본은 그대로 두고 추천용 DataFrame을 복사합니다.
df_reco = df_books.copy()

# 전처리 전 도서 수를 저장합니다.
rows_before = len(df_reco)

# 상품명을 문자열로 정리합니다.
df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 빈 상품명은 추천에 사용할 수 없으므로 제외합니다.
df_reco = (
    df_reco[df_reco["상품명"] != ""]
    .reset_index(drop=True)
)

rows_after = len(df_reco)

print("전처리 전 도서 수:", rows_before)
print("추천에 사용할 도서 수:", rows_after)
print("제외된 도서 수:", rows_before - rows_after)

df_reco.head()

In [ ]:
# index가 0부터 순서대로 연결되어 있는지 확인합니다.
print("앞의 index:", df_reco.index[:10].tolist())
print("마지막 index:", df_reco.index[-1])
print("도서 수 - 1:", len(df_reco) - 1)

print(
    "index가 0부터 도서 수-1까지 이어지는가?:",
    df_reco.index.equals(pd.RangeIndex(len(df_reco)))
)

### 실습 3 결과 확인 및 정리

`reset_index(drop=True)`는 이번 추천에서 특히 중요합니다.

뒤에서 TF-IDF 행렬은 다음처럼 만들어집니다.

```text
matrix row 0 ↔ df_reco index 0
matrix row 1 ↔ df_reco index 1
matrix row 2 ↔ df_reco index 2
...
```

이 대응이 틀어지면 **A책의 유사도 점수를 B책의 정보와 연결하는 오류**가 생길 수 있습니다.

따라서 추천용 데이터를 필터링한 뒤 index를 다시 0부터 정리합니다.

## 실습 4. 콘텐츠 기반 추천 이해하기

### AI에게 질문

> 콘텐츠 기반 추천을 처음 배우고 있습니다.  
> 이번에는 사용자 이력이 아니라 도서 제목만 이용합니다.
>
> 콘텐츠 기반 추천이 무엇인지, 그리고 이번 실습이 개인화 추천과 어떻게 다른지 초보자에게 설명해 주세요.

### AI 답변

콘텐츠 기반 추천은 **선택한 항목의 특징과 비슷한 특징을 가진 다른 항목을 찾는 방식**입니다.

이번 실습에서 사용할 특징은 도서의 **상품명 텍스트**입니다.

```text
도서 A 제목 → TF-IDF 벡터 A
도서 B 제목 → TF-IDF 벡터 B

A와 B 벡터의 방향이 비슷한가?
        ↓
비슷하면 추천 후보
```

이번 추천은 다음 정보를 사용하지 않습니다.

- 사용자 구매 이력
- 클릭 이력
- 평점
- 판매량
- 개인 취향

따라서 정확한 표현은 **도서 제목 기반 유사 도서 추천**입니다.

### 실습 4 결과 확인 및 정리

이번 추천 결과에서 유사도가 높다는 것은 **제목의 TF-IDF 표현이 비슷하다**는 뜻입니다.

다음과 같은 의미는 아닙니다.

- 사용자가 반드시 좋아한다.
- 더 좋은 책이다.
- 판매량이 더 높다.
- 구매 가능성이 더 높다.

추천 시스템의 입력 정보가 무엇인지 정확히 알고 결과의 의미를 제한해서 해석해야 합니다.

## 실습 5. 코사인 유사도 이해하기

### AI에게 질문

> 코사인 유사도를 처음 배우고 있습니다.  
> [1, 1], [2, 2], [1, 0] 세 벡터를 이용해 cosine_similarity 결과를 보여 주세요.
>
> 1에 가까운 값과 0에 가까운 값의 의미, 자기 자신과의 유사도가 왜 1인지도 설명해 주세요.

### AI 답변

코사인 유사도는 두 벡터의 **크기 자체보다 방향이 얼마나 비슷한지** 비교합니다.

In [ ]:
# 아주 작은 벡터 세 개를 만듭니다.
vectors = np.array([
    [1, 1],
    [2, 2],
    [1, 0],
])

# 모든 벡터끼리의 코사인 유사도를 계산합니다.
vector_similarity = cosine_similarity(vectors)

vector_similarity

In [ ]:
# 사람이 읽기 쉽게 표로 바꿉니다.
vector_names = ["[1,1]", "[2,2]", "[1,0]"]

vector_similarity_df = pd.DataFrame(
    vector_similarity,
    index=vector_names,
    columns=vector_names,
)

vector_similarity_df.round(4)

### 실습 5 결과 확인 및 정리

`[1,1]`과 `[2,2]`는 크기는 다르지만 같은 방향을 가리키므로 코사인 유사도가 매우 높습니다.

기본적으로:

- **1에 가까움** → 현재 벡터 표현에서 방향이 매우 비슷함
- **0에 가까움** → 공통 특징이 적음
- **자기 자신** → 방향이 완전히 같으므로 일반적으로 1

따라서 도서 추천에서 아무 처리 없이 가장 높은 유사도 순으로 가져오면 **선택한 책 자기 자신이 1등**으로 나옵니다.

뒤에서 반드시 자기 자신을 제외합니다.

## 실습 6. 실제 도서 제목을 TF-IDF로 변환하기

### AI에게 질문

> df_reco의 상품명 전체를 TfidfVectorizer로 변환하고 싶습니다.
>
> 다음을 확인해 주세요.
>
> 1. 제목 수  
> 2. TF-IDF 행렬의 도서 수  
> 3. feature 단어 수  
> 4. 제목 수와 행렬 행 수가 같은지  
> 5. 처음 20개 feature
>
> 왜 Chapter 04와 달리 이번에는 전체 추천 대상에 fit_transform을 해도 되는지도 설명해 주세요.

### AI 답변

이번 Chapter에서는 현재 카탈로그 안에서 서로 비슷한 도서를 찾는 것이 목적이므로 **현재 추천 대상 전체를 하나의 공통 TF-IDF 공간으로 표현**합니다.

In [ ]:
# 추천에 사용할 제목 Series입니다.
titles = df_reco["상품명"]

# 전체 추천 대상 제목을 하나의 공통 TF-IDF 공간으로 만듭니다.
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(titles)

feature_names = tfidf.get_feature_names_out()

print("제목 수:", len(titles))
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])

print(
    "제목 수 == TF-IDF 행 수:",
    len(titles) == tfidf_matrix.shape[0]
)

print("\n처음 20개 feature:")
print(feature_names[:20])

In [ ]:
# TF-IDF가 sparse matrix인지 확인합니다.
print("행렬 타입:", type(tfidf_matrix))
print("0이 아닌 값 수:", tfidf_matrix.nnz)

# 전체 셀 중 0의 비율도 확인합니다.
total_cells = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
zero_ratio = 1 - (tfidf_matrix.nnz / total_cells)

print("전체 셀 수:", total_cells)
print("0의 비율:", round(zero_ratio, 4))

### 실습 6 결과 확인 및 정리

행과 열의 의미는 다음과 같습니다.

- **행** → 도서
- **열** → 단어
- **값** → 해당 제목의 TF-IDF 가중치

Chapter 04에서는 **모델 성능 평가**가 목적이어서 train/test를 먼저 나눈 뒤 train에만 TF-IDF를 fit했습니다.

이번 Chapter에서는 현재 보유한 추천 카탈로그 안에서 서로의 유사도를 비교하는 것이 목적입니다. 따라서 현재 추천 대상 전체를 같은 TF-IDF 공간으로 표현합니다.

나중에 추천 시스템을 운영하면서 새 도서를 추가하는 방식까지 설계한다면 Vectorizer를 언제 다시 fit할지 별도로 고민해야 합니다.

## 실습 7. 기준 도서 선택하기

### AI에게 질문

> df_reco에서 index 0번 도서를 기준 도서로 선택하고 싶습니다.
>
> index가 유효한지 확인하고, 실제 상품명과 가능한 경우 저자/출판사/분야 정보도 같이 확인하는 코드를 작성해 주세요.

### AI 답변

추천에서는 숫자 index만 보고 넘어가지 않고 **그 index가 실제 어떤 책인지 반드시 확인**합니다.

In [ ]:
# 기준 도서 index입니다.
selected_index = 0

# index 범위를 먼저 확인합니다.
if not 0 <= selected_index < len(df_reco):
    raise IndexError(f"유효하지 않은 index입니다: {selected_index}")

selected_title = df_reco.loc[
    selected_index,
    "상품명",
]

print("선택 index:", selected_index)
print("선택 도서:", selected_title)

# 선택 도서의 부가 정보도 가능한 만큼 확인합니다.
selected_columns = [
    col
    for col in ["상품명", "저자", "인물", "출판사", "분야"]
    if col in df_reco.columns
]

display(df_reco.loc[[selected_index], selected_columns])

### 실습 7 결과 확인 및 정리

추천의 기준점이 되는 책을 한 권 선택했습니다.

앞으로 계산하는 모든 유사도는 **이 선택 도서와 다른 도서가 얼마나 비슷한가**를 뜻합니다.

index 번호만 기억하지 말고 실제 제목을 같이 확인해야 이후 추천 결과가 자연스러운지 판단할 수 있습니다.

## 실습 8. 선택 도서와 전체 도서의 유사도 계산하기

### AI에게 질문

> 선택한 도서의 TF-IDF 벡터 한 행을 가져와 전체 도서와 cosine_similarity를 계산하고 싶습니다.
>
> 다음도 확인해 주세요.
>
> 1. 선택 벡터 shape  
> 2. 유사도 점수 개수  
> 3. 전체 도서 수  
> 4. 자기 자신과의 유사도  
> 5. similarity_scores[index]와 df_reco.iloc[index]가 대응하는 구조 설명

### AI 답변

선택한 한 행과 전체 행렬을 비교하면 선택 도서와 모든 도서의 유사도가 한 번에 계산됩니다.

In [ ]:
# 선택 도서의 TF-IDF 벡터 한 행을 가져옵니다.
selected_vector = tfidf_matrix[selected_index]

print("선택 벡터 shape:", selected_vector.shape)

# 선택 도서와 전체 도서의 코사인 유사도를 계산합니다.
# 결과가 (1, 도서수) 형태이므로 flatten()으로 1차원으로 바꿉니다.
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix,
).flatten()

print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))
print(
    "개수가 같은가?:",
    len(similarity_scores) == len(df_reco)
)

print(
    "자기 자신과의 유사도:",
    similarity_scores[selected_index]
)

In [ ]:
# 앞의 몇 개 index가 실제 어떤 책과 연결되는지 직접 확인합니다.
mapping_check = pd.DataFrame({
    "index": np.arange(min(10, len(df_reco))),
    "상품명": df_reco["상품명"].head(10).to_list(),
    "similarity": similarity_scores[:10],
})

mapping_check

### 실습 8 결과 확인 및 정리

`similarity_scores[0]`은 `df_reco.iloc[0]`, `similarity_scores[1]`은 `df_reco.iloc[1]`에 대응합니다.

이 대응이 정확하기 때문에 실습 3에서 index를 0부터 다시 맞춘 것입니다.

선택 도서 자기 자신의 유사도는 일반적으로 **1에 매우 가깝게** 나옵니다.

## 실습 9. 유사도가 높은 순서 확인하기

### AI에게 질문

> similarity_scores를 상품명과 같이 DataFrame으로 만들고, 유사도가 높은 순서로 상위 10개를 확인하고 싶습니다.
>
> 가장 위에 자기 자신이 나오는지도 확인하는 코드를 작성해 주세요.

### AI 답변

먼저 자기 자신을 포함한 원래 순위를 확인하면 이후 '자기 자신 제외' 로직이 왜 필요한지 직접 볼 수 있습니다.

In [ ]:
# index, 상품명, 유사도를 하나의 표로 만듭니다.
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"].to_numpy(),
    "similarity": similarity_scores,
})

# 유사도가 높은 순서로 정렬합니다.
score_sorted = (
    score_df
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

score_sorted.head(10)

In [ ]:
# 가장 높은 결과가 자기 자신인지 확인합니다.
top_row = score_sorted.iloc[0]

print("1위 index:", int(top_row["index"]))
print("선택 index:", selected_index)
print("1위 제목:", top_row["상품명"])
print("선택 제목:", selected_title)
print("1위가 자기 자신인가?:", int(top_row["index"]) == selected_index)

### 실습 9 결과 확인 및 정리

가장 위에는 일반적으로 선택 도서 자기 자신이 나옵니다.

```text
선택 도서 ↔ 선택 도서
similarity ≈ 1.0
```

이것은 오류가 아니라 정상적인 결과입니다.

추천 목록의 목적은 **선택한 책과 비슷한 다른 책**을 찾는 것이므로 다음 단계에서 자기 자신을 제외합니다.

## 실습 10. 자기 자신을 제외하고 Top 5 만들기

### AI에게 질문

> similarity_scores를 높은 순서로 정렬한 뒤
> 1. 자기 자신을 제외하고
> 2. 상위 5개 index를 선택하고
> 3. 상품명과 유사도를 표로 만들고
> 4. 실제로 5개 이하인지, 자기 자신이 없는지, 유사도 내림차순인지 검증
>
> 하는 코드를 작성해 주세요.

### AI 답변

추천의 핵심 로직은 **유사도 계산 → 높은 순서 정렬 → 자기 자신 제외 → Top N**입니다.

In [ ]:
# 유사도 점수의 index를 높은 순서로 정렬합니다.
sorted_indices = similarity_scores.argsort()[::-1]

# 자기 자신을 제외하고 상위 5개 index를 선택합니다.
recommended_indices = [
    int(idx)
    for idx in sorted_indices
    if int(idx) != selected_index
][:5]

print("추천 index:", recommended_indices)

# 추천 결과에 사용할 컬럼을 실제 존재하는 컬럼 기준으로 고릅니다.
result_columns = [
    col
    for col in ["상품명", "저자", "인물", "출판사", "분야"]
    if col in df_reco.columns
]

# 추천 도서 정보를 가져옵니다.
result = df_reco.loc[
    recommended_indices,
    result_columns,
].copy()

# 각 추천 도서의 유사도를 붙입니다.
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

# index는 보기 좋게 다시 정리합니다.
result = result.reset_index(drop=True)

result

In [ ]:
# 추천 로직이 제대로 작동했는지 검증합니다.
print("추천 결과 수:", len(result))
print("5개 이하인가?:", len(result) <= 5)
print(
    "자기 자신이 추천 index에 없는가?:",
    selected_index not in recommended_indices
)

# 유사도가 내림차순인지 확인합니다.
is_descending = all(
    result["similarity"].iloc[i]
    >= result["similarity"].iloc[i + 1]
    for i in range(len(result) - 1)
)

print("유사도 내림차순인가?:", is_descending)

### 실습 10 결과 확인 및 정리

추천의 가장 기본적인 로직이 완성되었습니다.

```text
유사도 계산
→ 높은 순서 정렬
→ 자기 자신 제외
→ Top 5
```

여기서는 자기 자신만 제외했습니다.

뒤의 추천 함수에서는 **같은 제목이 중복 등록된 경우**도 함께 제외하도록 조금 더 안전하게 만듭니다.

## 실습 11. 추천 결과에 메타데이터 추가하기

제목만 보여 주면 추천 결과가 실제로 자연스러운지 판단하기 어렵습니다.

가능한 경우 저자, 출판사, 분야 정보를 같이 확인합니다.

In [ ]:
# 선택 도서 정보를 먼저 확인합니다.
print("선택 도서:")
display(df_reco.loc[[selected_index], result_columns])

print("\n추천 도서:")
display(result)

### 실습 11 직접 확인할 질문

추천 결과를 실제로 읽으면서 다음을 확인합니다.

- 제목에 공통 단어가 있는가?
- 같은 시리즈 또는 비슷한 제목 구조인가?
- 분야도 비슷한가?
- 분야는 다른데 제목 표현만 비슷한 것은 아닌가?
- 유사도 값이 높은 추천과 낮은 추천의 제목 차이가 실제로 느껴지는가?

추천 시스템에서는 숫자만 보는 것보다 **실제 추천 결과를 사람이 읽어 보는 검증**이 중요합니다.

## 실습 12. 추천 함수 만들기

### AI에게 질문

> 지금까지 만든 추천 코드를 `recommend_books()` 함수로 묶고 싶습니다.
>
> 조건:
> - selected_index
> - df
> - tfidf_matrix
> - top_n=5
> - index 범위 검사
> - 자기 자신 제외
> - 동일한 상품명 제외
> - 실제 존재하는 상품명/저자/인물/출판사/분야 컬럼만 반환
> - similarity 추가
> - 유사도 높은 순서 유지
> - 결과 index 다시 정리
>
> 초보자가 이해하도록 주석을 자세히 달아 주세요.

### AI 답변

함수는 반복해서 추천을 실행하기 위한 도구입니다. 선택 도서를 받아 전체 도서와 유사도를 계산하고, 자기 자신과 동일 제목을 제외한 뒤 Top N을 반환합니다.

In [ ]:
def recommend_books(
    selected_index,
    df,
    tfidf_matrix,
    top_n=5,
):
    # top_n은 1 이상의 정수여야 합니다.
    if top_n < 1:
        raise ValueError("top_n은 1 이상이어야 합니다.")

    # DataFrame index와 TF-IDF 행 번호가 같은 구조인지 먼저 확인합니다.
    if len(df) != tfidf_matrix.shape[0]:
        raise ValueError(
            "DataFrame 행 수와 TF-IDF 행 수가 다릅니다. "
            "추천 데이터와 행렬의 대응을 확인하세요."
        )

    # selected_index가 실제 범위 안에 있는지 확인합니다.
    if not 0 <= selected_index < len(df):
        raise IndexError(
            f"유효하지 않은 index입니다: {selected_index}"
        )

    # 선택한 도서의 실제 제목을 가져옵니다.
    selected_title = df.loc[
        selected_index,
        "상품명",
    ]

    # 선택한 도서와 전체 도서의 코사인 유사도를 계산합니다.
    scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix,
    ).flatten()

    # 유사도 높은 순서의 index를 구합니다.
    sorted_indices = scores.argsort()[::-1]

    # 최종 추천 index를 담을 빈 리스트입니다.
    recommended_indices = []

    for idx in sorted_indices:
        idx = int(idx)

        # 1. 선택한 자기 자신은 추천에서 제외합니다.
        if idx == selected_index:
            continue

        # 2. 자기 자신과 상품명이 완전히 같은 중복 제목도 제외합니다.
        if df.loc[idx, "상품명"] == selected_title:
            continue

        # 조건을 통과한 도서를 추천 후보에 추가합니다.
        recommended_indices.append(idx)

        # 원하는 개수를 채우면 반복을 종료합니다.
        if len(recommended_indices) >= top_n:
            break

    # 실제 데이터에 존재하는 메타데이터 컬럼만 사용합니다.
    columns = [
        column
        for column in [
            "상품명",
            "저자",
            "인물",
            "출판사",
            "분야",
        ]
        if column in df.columns
    ]

    # 추천 도서 정보를 가져옵니다.
    result = df.loc[
        recommended_indices,
        columns,
    ].copy()

    # 유사도 값을 추가합니다.
    result["similarity"] = [
        round(float(scores[idx]), 4)
        for idx in recommended_indices
    ]

    # 보기 좋게 index를 다시 0부터 설정합니다.
    return result.reset_index(drop=True)

### 실습 12 함수 흐름 정리

이 함수가 하는 일은 다음과 같습니다.

```text
선택한 도서 index 받기
→ index/행렬 구조 검증
→ 전체 도서와 유사도 계산
→ 높은 순서 정렬
→ 자기 자신 제외
→ 동일 제목 제외
→ Top N 선택
→ 메타데이터 + similarity 반환
```

함수 안에서 **데이터 행 수와 TF-IDF 행 수가 같은지 검증**하도록 넣어, index 대응이 깨진 상태에서 잘못된 추천이 만들어지는 것을 막았습니다.

## 실습 13. 추천 함수 실행하기

### AI에게 질문

> recommend_books()를 index 0에 대해 top_n=5로 실행하고,
> 자기 자신이 빠졌는지, 결과 수가 5개 이하인지, similarity가 내림차순인지 검증해 주세요.

### AI 답변

함수 결과만 출력하지 않고 추천 구조가 실제로 조건을 만족하는지 함께 검증합니다.

In [ ]:
# 기준 도서를 다시 선택합니다.
selected_index = 0

print(
    "선택 도서:",
    df_reco.loc[selected_index, "상품명"],
)

# 추천 함수를 실행합니다.
recommendations = recommend_books(
    selected_index=selected_index,
    df=df_reco,
    tfidf_matrix=tfidf_matrix,
    top_n=5,
)

recommendations

In [ ]:
# DataFrame과 TF-IDF 행 수가 같은지 확인합니다.
assert df_reco.shape[0] == tfidf_matrix.shape[0]

# 추천 결과 수 확인
print("추천 결과 수:", len(recommendations))
print("5개 이하인가?:", len(recommendations) <= 5)

# 동일 제목이 추천에 섞였는지 확인
selected_title = df_reco.loc[selected_index, "상품명"]

same_title_in_result = (
    recommendations["상품명"] == selected_title
).any()

print("동일 제목이 추천 결과에 있는가?:", same_title_in_result)

# 유사도 내림차순 확인
is_descending = all(
    recommendations["similarity"].iloc[i]
    >= recommendations["similarity"].iloc[i + 1]
    for i in range(len(recommendations) - 1)
)

print("유사도 내림차순인가?:", is_descending)

### 실습 13 결과 확인 및 정리

추천 함수가 작동한다고 판단하려면 최소한 다음 네 가지를 확인합니다.

- 자기 자신이 제외되었는가?
- 동일 제목이 제외되었는가?
- 결과가 최대 Top N개인가?
- similarity가 높은 순서인가?

그리고 가장 중요한 검증은 **사람이 실제 제목을 읽는 것**입니다.

수치적으로 가장 유사한 결과라도 제목 기반이라는 한계 때문에 사람이 보기에 어색할 수 있습니다.

## 실습 14. 추천 결과 저장하기

### AI에게 질문

> recommendations를 `chapter05_recommendations.csv`로 저장하고 싶습니다.
>
> 프로젝트의 `notebooks/book-text-ml` 폴더에 utf-8-sig로 저장하고,
> 파일 존재 여부, 다시 읽은 행 수, 앞부분까지 확인하게 작성해 주세요.

### AI 답변

저장 명령 실행만으로 끝내지 않고 실제 결과 파일을 다시 읽어 검증합니다.

In [ ]:
# 추천 결과 저장 경로입니다.
RECOMMENDATION_PATH = Path(
    "notebooks/book-text-ml/chapter05_recommendations.csv"
)

# CSV로 저장합니다.
recommendations.to_csv(
    RECOMMENDATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("추천 결과 파일 존재:", RECOMMENDATION_PATH.exists())
print("저장 경로:", RECOMMENDATION_PATH)

In [ ]:
# 저장한 파일을 다시 읽어 정상적으로 저장되었는지 확인합니다.
recommendations_check = pd.read_csv(
    RECOMMENDATION_PATH,
    encoding="utf-8-sig",
)

print("저장된 행 수:", len(recommendations_check))
display(recommendations_check)

### 실습 14 결과 확인 및 정리

파일을 저장한 뒤 반드시 다음을 확인합니다.

- 파일이 실제로 생성되었는가?
- 한글이 정상적으로 보이는가?
- 추천 결과 수가 원래 DataFrame과 같은가?
- 상품명과 similarity 값이 정상인가?

이번 결과물은 다음 Chapter의 Streamlit 화면에서도 활용할 수 있습니다.

## 실습 15. 추천 결과의 한계 이해하기

현재 추천은 **상품명 텍스트만 사용**합니다.

따라서 다음과 같은 현상이 생길 수 있습니다.

```text
같은 분야라도 제목 단어가 다름
→ 유사도가 낮을 수 있음

다른 분야라도 제목 표현이 비슷함
→ 유사도가 높을 수 있음
```

또한 코사인 유사도는 사용자 취향 점수가 아닙니다.

```text
유사도 높음
≠ 사용자가 반드시 좋아함
≠ 더 좋은 책
≠ 구매 가능성이 높음
```

### 권장 표현

> 현재 TF-IDF 제목 표현 기준으로 유사도가 높은 도서입니다.

### 피해야 할 표현

> 사용자가 가장 좋아할 책입니다.

이번 추천은 **제목 기반 콘텐츠 유사도 추천**이라는 범위에서 해석합니다.

## 실습 16. Chapter 06 Streamlit 연결 준비하기

### AI에게 질문

> 다음 Chapter에서 Streamlit selectbox로 책을 선택하고 recommend_books() 결과를 보여 주려고 합니다.
>
> df_reco의 index와 상품명을 함께 표시하는 선택 옵션 dictionary를 만들어 주세요.
> 선택 문자열은 "index | 상품명" 형태로 만들고 값에는 실제 index를 저장해 주세요.

### AI 답변

제목만 사용하면 같은 제목이 여러 개 있을 때 구분하기 어렵기 때문에 index와 제목을 함께 표시합니다.

In [ ]:
# Streamlit selectbox에서 사용할 선택 목록을 만듭니다.
book_options = {
    f"{idx} | {row['상품명']}": idx
    for idx, row in df_reco.iterrows()
}

# 앞의 10개 옵션을 확인합니다.
list(book_options.items())[:10]

In [ ]:
# 실제로 선택 문자열 하나를 index로 되돌리는 흐름을 확인합니다.
first_option_text = next(iter(book_options))

selected_index_from_option = book_options[first_option_text]

print("화면에 표시될 문자열:", first_option_text)
print("실제 추천에 사용할 index:", selected_index_from_option)

display(
    recommend_books(
        selected_index=selected_index_from_option,
        df=df_reco,
        tfidf_matrix=tfidf_matrix,
        top_n=5,
    )
)

### 실습 16 결과 확인 및 정리

Chapter 06의 예상 흐름은 다음과 같습니다.

```text
st.selectbox()
      ↓
사용자가 "index | 상품명" 선택
      ↓
selected_index
      ↓
recommend_books()
      ↓
Top 5
      ↓
st.dataframe()
```

Chapter 06에서는 새로운 추천 알고리즘을 다시 만드는 것이 아니라 **이번 Chapter에서 완성한 함수와 데이터를 UI에 연결**합니다.

## 실습 17. 전체 코드 흐름 정리하기

이번 셀은 새 내용을 추가하는 것이 아니라 앞에서 배운 전체 추천 흐름을 한 번에 복습하기 위한 코드입니다.

**CSV 로드 → 상품명 정리 → index 재설정 → TF-IDF → 기준 도서 → cosine similarity → 자기 자신/동일 제목 제외 → Top 5 → 저장**

In [ ]:
# 1. 데이터 불러오기
df_reco_check = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 2. 상품명 정리
df_reco_check["상품명"] = (
    df_reco_check["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_reco_check = (
    df_reco_check[df_reco_check["상품명"] != ""]
    .reset_index(drop=True)
)

# 3. TF-IDF
tfidf_check = TfidfVectorizer()

tfidf_matrix_check = tfidf_check.fit_transform(
    df_reco_check["상품명"]
)

# 4. 같은 구조의 추천 함수
def recommend_books_check(
    selected_index,
    df,
    matrix,
    top_n=5,
):
    if not 0 <= selected_index < len(df):
        raise IndexError(
            f"유효하지 않은 index입니다: {selected_index}"
        )

    if len(df) != matrix.shape[0]:
        raise ValueError(
            "DataFrame과 TF-IDF 행 수가 다릅니다."
        )

    selected_title = df.loc[selected_index, "상품명"]

    scores = cosine_similarity(
        matrix[selected_index],
        matrix,
    ).flatten()

    sorted_indices = scores.argsort()[::-1]

    recommended_indices = []

    for idx in sorted_indices:
        idx = int(idx)

        if idx == selected_index:
            continue

        if df.loc[idx, "상품명"] == selected_title:
            continue

        recommended_indices.append(idx)

        if len(recommended_indices) >= top_n:
            break

    columns = [
        col
        for col in [
            "상품명",
            "저자",
            "인물",
            "출판사",
            "분야",
        ]
        if col in df.columns
    ]

    output = df.loc[
        recommended_indices,
        columns,
    ].copy()

    output["similarity"] = [
        round(float(scores[idx]), 4)
        for idx in recommended_indices
    ]

    return output.reset_index(drop=True)

# 5. 추천 실행
recommendations_check = recommend_books_check(
    selected_index=0,
    df=df_reco_check,
    matrix=tfidf_matrix_check,
    top_n=5,
)

recommendations_check

In [ ]:
# 앞에서 만든 결과와 전체 흐름 요약 결과가 같은지 확인합니다.
print(
    "추천용 도서 수 동일:",
    len(df_reco) == len(df_reco_check)
)

print(
    "TF-IDF shape 동일:",
    tfidf_matrix.shape == tfidf_matrix_check.shape
)

print(
    "추천 상품명 동일:",
    recommendations["상품명"].tolist()
    == recommendations_check["상품명"].tolist()
)

print(
    "추천 유사도 동일:",
    recommendations["similarity"].tolist()
    == recommendations_check["similarity"].tolist()
)

### 실습 17 결과 확인 및 정리

같은 데이터와 같은 Vectorizer 설정을 사용했기 때문에 앞에서 단계별로 만든 결과와 마지막 요약 코드의 결과가 같아야 합니다.

코드를 외우기보다 각 단계가 무엇을 하는지 순서대로 설명할 수 있어야 합니다.

1. 상품명을 정리한다.
2. DataFrame index를 TF-IDF 행 번호와 맞춘다.
3. 모든 추천 대상 제목을 TF-IDF 벡터로 바꾼다.
4. 기준 도서 한 권을 고른다.
5. 기준 벡터와 전체 벡터의 코사인 유사도를 계산한다.
6. 유사도를 높은 순서로 정렬한다.
7. 자기 자신과 동일 제목을 제외한다.
8. Top N을 반환한다.

## 실제 결과를 이용한 Chapter 05 Markdown 정리

아래 셀은 임의의 추천 제목이나 유사도 숫자를 적지 않고 **현재 Notebook에서 실제 계산된 추천 결과**를 이용해 Markdown을 만듭니다.

In [ ]:
# 실제 선택 도서와 추천 결과를 Markdown 문자열로 만듭니다.
selected_title = df_reco.loc[selected_index, "상품명"]

recommendation_lines = []

for rank, row in recommendations.iterrows():
    recommendation_lines.append(
        f"- {rank + 1}위: **{row['상품명']}** "
        f"(유사도: {row['similarity']:.4f})"
    )

recommendation_md = "\n".join(recommendation_lines)

chapter05_md = f"""
## Chapter 05 결과

### 추천 기준 도서

- **{selected_title}**

### 추천 방식

- 도서 제목을 **TfidfVectorizer**로 숫자 벡터로 변환했습니다.
- 기준 도서와 전체 도서 사이의 **Cosine Similarity**를 계산했습니다.
- 기준 도서 자기 자신과 동일 제목을 제외했습니다.
- 유사도가 높은 순서로 최대 5권을 선택했습니다.

### 추천 결과

{recommendation_md}

### 해석

위 결과는 **현재 TF-IDF 제목 표현 기준으로 유사도가 높은 도서**입니다.
유사도가 높다고 해서 사용자가 반드시 좋아하거나 더 좋은 책이라는 뜻은 아닙니다.

### 한계

- 현재 추천은 주로 **도서 제목 텍스트만 사용**합니다.
- 사용자 구매 이력, 평점, 클릭 정보는 사용하지 않습니다.
- 같은 분야라도 제목 단어가 다르면 낮은 유사도가 나올 수 있습니다.
- 다른 분야라도 제목 표현이 비슷하면 높은 유사도가 나올 수 있습니다.
"""

display(Markdown(chapter05_md))

### Chapter 05 결과 Markdown 확인

이 Markdown은 실제 `recommendations` 결과에서 상품명과 유사도를 가져옵니다.

따라서 아직 실행하지 않은 숫자나 추천 제목을 미리 만들어 적지 않습니다.

**Kernel Restart → Run All**을 했을 때 현재 데이터 기준 결과가 자동으로 반영됩니다.

## 최종 체크리스트

Notebook을 제출하거나 다음 Chapter로 넘어가기 전에 처음부터 다시 실행합니다.

- [ ] 전처리 데이터를 불러왔다.
- [ ] 상품명을 정리했다.
- [ ] 추천용 DataFrame index를 0부터 다시 설정했다.
- [ ] DataFrame index와 TF-IDF matrix 행 번호의 대응을 이해했다.
- [ ] 콘텐츠 기반 추천의 의미를 설명할 수 있다.
- [ ] 간단한 벡터로 코사인 유사도를 확인했다.
- [ ] 도서 제목을 TF-IDF 벡터로 변환했다.
- [ ] 선택 도서와 전체 도서의 유사도를 계산했다.
- [ ] 유사도 개수와 전체 도서 수가 같은지 확인했다.
- [ ] 유사도를 높은 순서로 정렬했다.
- [ ] 자기 자신을 제외했다.
- [ ] 동일 제목도 제외했다.
- [ ] Top 5 추천 결과를 만들었다.
- [ ] 추천 함수를 만들었다.
- [ ] 추천 결과의 정렬과 개수를 검증했다.
- [ ] 추천 결과를 실제 제목과 비교했다.
- [ ] 결과를 CSV로 저장하고 다시 읽어 확인했다.
- [ ] Streamlit selectbox 연결용 옵션을 만들었다.
- [ ] 제목 기반 추천의 한계를 설명할 수 있다.
- [ ] 실제 실행 결과를 이용한 Markdown을 작성했다.

## 이번 Chapter에서 꼭 기억할 5가지

1. **TF-IDF 벡터는 분류뿐 아니라 추천에도 사용할 수 있습니다.**
2. **코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 비교합니다.**
3. **자기 자신과의 유사도가 가장 높으므로 추천에서 제외합니다.**
4. **유사도 내림차순 → 자기 자신/동일 제목 제외 → Top N이 기본 추천 흐름입니다.**
5. **제목 유사도는 사용자의 취향이나 책의 품질을 의미하지 않습니다.**

### 다음 Chapter 연결

지금까지 두 가지 기능을 만들었습니다.

```text
Chapter 04
새 도서 제목
→ TF-IDF
→ Naive Bayes
→ 예상 분야

Chapter 05
기존 도서 선택
→ TF-IDF
→ Cosine Similarity
→ 유사 도서 Top 5
```

Chapter 06에서는 이 두 기능을 Streamlit UI에 연결합니다.

### Chapter 05 한 문장 정리

**도서 제목을 TF-IDF 벡터로 표현하고, 선택한 도서와 전체 도서의 코사인 유사도를 계산한 뒤 자기 자신과 동일 제목을 제외하고 유사도가 높은 Top 5를 선택하면 제목 기반 콘텐츠 추천을 만들 수 있습니다.**